In [1]:
# ============================================================================
# Cell 1 — Package installation
# ============================================================================
import os, sys, subprocess

def _pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"pip install failed for {pkgs}")

_pip_install("easyfsl==1.5.0", "gdown")
print("Packages ready.")

Packages ready.


In [2]:
# ============================================================================
# Cell 2 — MiniImageNet download + extraction  (unchanged from master)
# ============================================================================
import tarfile, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FOLDER   = "https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM"
DOWNLOAD_DIR   = WORK_DIR / "miniimagenet_archives"
DATA_DIR       = WORK_DIR / "miniimagenet"
EXPECTED_CLASSES = {"train": 64, "val": 16, "test": 20}

def _split_ready(split):
    d = DATA_DIR / split / split
    return d.is_dir() and sum(1 for p in d.iterdir() if p.is_dir()) == EXPECTED_CLASSES[split]

if all(_split_ready(s) for s in EXPECTED_CLASSES):
    print("MiniImageNet already extracted; skipping download.")
else:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, 4):
        result = subprocess.run(
            [sys.executable, "-m", "gdown", "--folder", DRIVE_FOLDER, "-O", str(DOWNLOAD_DIR)],
            check=False)
        print("Download exit code:", result.returncode, f"(attempt {attempt}/3)")
        if all(list(DOWNLOAD_DIR.rglob(f"{s}.tar")) for s in EXPECTED_CLASSES):
            break
    print("Downloaded files:")
    for p in DOWNLOAD_DIR.rglob("*"):
        if p.is_file():
            print(f"  {p.name}: {p.stat().st_size / 1e6:.1f} MB")
    for split in ("train", "val", "test"):
        matches = list(DOWNLOAD_DIR.rglob(f"{split}.tar"))
        if not matches:
            raise FileNotFoundError(
                f"{split}.tar not found. Check Drive folder is public, "
                "Internet is enabled, and download quota is not exhausted.")
        archive = matches[0]
        out_dir = DATA_DIR / split
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {archive.name} -> {out_dir} ...")
        with tarfile.open(archive, "r:*") as tar:
            try:
                tar.extractall(out_dir, filter="data")
            except TypeError:
                tar.extractall(out_dir)
        print(f"Finished extracting {split}")

for s in EXPECTED_CLASSES:
    if not _split_ready(s):
        raise RuntimeError(
            f"Split '{s}' not correctly extracted "
            f"(expected {EXPECTED_CLASSES[s]} class folders in {DATA_DIR / s / s}).")
print("Dataset ready.")

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v
To: /kaggle/working/miniimagenet_archives/test.tar
 17%|█▋        | 6.82M/39.2M [00:00<00:00, 45.8MB/s]

Processing file 1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v test.tar
Processing file 107FTosYIeBn5QbynR46YG91nHcJ70whs train.tar
Processing file 1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl val.tar


100%|██████████| 39.2M/39.2M [00:00<00:00, 49.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs
From (redirected): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs&confirm=t&uuid=93bf166f-38ef-4b65-8353-9e4060255a35
To: /kaggle/working/miniimagenet_archives/train.tar
100%|██████████| 126M/126M [00:01<00:00, 65.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl
To: /kaggle/working/miniimagenet_archives/val.tar
100%|██████████| 30.8M/30.8M [00:00<00:00, 195MB/s]
Download completed


Download exit code: 0 (attempt 1/3)
Downloaded files:
  train.tar: 125.9 MB
  test.tar: 39.2 MB
  val.tar: 30.8 MB
Extracting train.tar -> /kaggle/working/miniimagenet/train ...
Finished extracting train
Extracting val.tar -> /kaggle/working/miniimagenet/val ...
Finished extracting val
Extracting test.tar -> /kaggle/working/miniimagenet/test ...
Finished extracting test
Dataset ready.


In [3]:
# ============================================================================
# Cell 3 — Imports, configuration, and memory helpers  (single configuration: DW + LayerNorm + EMA)
# ============================================================================
import os, gc, io, time, copy, json, math, random, shutil, hashlib, ctypes
import platform, multiprocessing, contextlib
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor, get_graph_node_names
from easyfsl.samplers import TaskSampler
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

try:
    import psutil
except ImportError:
    psutil = None

try:
    multiprocessing.set_start_method("spawn", force=True)
except RuntimeError:
    pass

# ── Seeds ─────────────────────────────────────────────────────────────────────
SEED               = 42
TRAIN_EPISODE_SEED = 20263
VAL_EPISODE_SEED   = 20261
TEST_EPISODE_SEED  = 20262

# ── Baseline hyperparameters (unchanged from master) ──────────────────────────
IMAGE_SIZE           = 84
MAX_EPOCHS           = 10
N_TRAINING_EPISODES  = 360
N_VALIDATION_TASKS   = 600
N_TEST_TASKS         = 600
TRAIN_WAY            = 30
TEST_WAY             = 5
SHOT                 = 5
QUERY                = 15
FEATURE_CHUNK_SIZE   = 600
ATTENTION_CHUNK_SIZE = 600
USE_AMP              = True
STAGE_NODES          = ("maxpool", "layer1", "layer2", "layer3", "layer4")
STAGE_WEIGHTS        = (1.0, 1.1, 1.2, 1.3, 1.4)
USE_IMAGENET_PRETRAINED = True
NUM_WORKERS          = 0
gamma_init           = 0.2
weights_lr           = 0.005
backbone_lr          = 1e-4
AMP_ON               = bool(USE_AMP and torch.cuda.is_available())

# ── EMA hyperparameter (unchanged from master) ────────────────────────────────
EMA_BETA = 0.99

# ── Dynamic Weights (DW) hyperparameters — verbatim from msenet-dynamicweights ─
DYNAMIC_GATE_SCALE = 0.5     # max modulation (+/- 50%) around the global prior weights
FISHER_TEMP        = 2.0     # initial softmax temperature for support-set Fisher / query confidence
dynamic_gate_lr    = 1e-3    # learning rate of all DW parameters

# ── Experiment configuration ──────────────────────────────────────────────────
# Single configuration (the best one): Euclidean distance + Dynamic Weights (DW)
# + per-stage LayerNorm + EMA.  Global average pooling is used (NO GeM).
EXPERIMENT_CONFIGS = {
    "DW_LN_EMA": dict(description="DW + LayerNorm + EMA",
                      use_layernorm=True, use_ema=True, gammalearnable=True),
}
CONFIG_ORDER = ["DW_LN_EMA"]
KEY          = CONFIG_ORDER[0]
DIST_LABEL   = "Euclidean"

# Everything needed to rebuild the architecture from the saved .pth file
MODEL_CONFIG = dict(
    image_size=IMAGE_SIZE, stage_nodes=list(STAGE_NODES), stage_weights_init=list(STAGE_WEIGHTS),
    weightlearnable=True, gammalearnable=True, gamma_init=gamma_init,
    use_layernorm=True, dynamic_gate_scale=DYNAMIC_GATE_SCALE, fisher_temp_init=FISHER_TEMP,
    distance="euclidean", pooling="global_avg_pool", backbone="resnet18",
    feature_chunk_size=FEATURE_CHUNK_SIZE, attention_chunk_size=ATTENTION_CHUNK_SIZE,
)

# ── Data paths ────────────────────────────────────────────────────────────────
WORK_DIR  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working").resolve()
DATA_DIR  = WORK_DIR / "miniimagenet"
TRAIN_DIR = str(DATA_DIR / "train" / "train")
VAL_DIR   = str(DATA_DIR / "val"   / "val")
TEST_DIR  = str(DATA_DIR / "test"  / "test")

RESULT_DIR = WORK_DIR / "results"
COMP_DIR   = RESULT_DIR / "comparison_tables"
PARAM_DIR  = RESULT_DIR / "parameter_tracking"
PLOT_DIR   = RESULT_DIR / "plots"
MODEL_DIR  = RESULT_DIR / "model"
CKPT_PATH  = MODEL_DIR / f"msenet_dw_ln_ema_{TEST_WAY}way{SHOT}shot_best.pth"
for d in (COMP_DIR, PARAM_DIR, PLOT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── RAM helpers ───────────────────────────────────────────────────────────────
RAM_LIMIT_GB  = 29.0
RAM_WARN_FRAC = 0.80
RAM_FAIL_FRAC = 0.92

def rss_gb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
    except Exception:
        pass
    if psutil is not None:
        return psutil.Process().memory_info().rss / 2**30
    return float("nan")

def free_host_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass

def ram_guard(tag=""):
    rss = rss_gb()
    if rss > RAM_WARN_FRAC * RAM_LIMIT_GB:
        free_host_memory(); rss = rss_gb()
        print(f"[RAM] WARNING {tag}: RSS {rss:.1f} GB above {RAM_WARN_FRAC:.0%} of {RAM_LIMIT_GB:.0f} GB limit.")
        if rss > RAM_FAIL_FRAC * RAM_LIMIT_GB:
            raise MemoryError(f"Process RSS {rss:.1f} GB exceeds {RAM_FAIL_FRAC:.0%} of {RAM_LIMIT_GB:.0f} GB limit.")
    return rss

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: No GPU detected. Training will be extremely slow on CPU.")

ENV_INFO = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "distance": "euclidean",
}
try:
    import torchvision, easyfsl
    ENV_INFO["torchvision"] = torchvision.__version__
    ENV_INFO["easyfsl"] = getattr(easyfsl, "__version__", "unknown")
except Exception:
    pass

PRELOAD_THREADS  = min(16, max(4, (os.cpu_count() or 4) * 2))
resize_transform = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD  = (0.229, 0.224, 0.225)

print(f"Device: {device} | GPU: {ENV_INFO['gpu']}")
print(f"Configuration: [{KEY}] {EXPERIMENT_CONFIGS[KEY]['description']}  (no GeM)")
print(f"Environment: {ENV_INFO}")
print(f"[RAM] start: {rss_gb():.2f} GB (limit {RAM_LIMIT_GB:.0f} GB)")

Device: cuda | GPU: Tesla T4
Configuration: [DW_LN_EMA] DW + LayerNorm + EMA  (no GeM)
Environment: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'cuda_available': True, 'gpu': 'Tesla T4', 'distance': 'euclidean', 'torchvision': '0.26.0+cu130', 'easyfsl': '1.5.0'}
[RAM] start: 0.74 GB (limit 29 GB)


In [4]:
# ============================================================================
# Cell 4 — RAM-cached datasets (PreloadedSet, unchanged from master)
# ============================================================================

_mean_gpu = torch.tensor(IMG_MEAN, dtype=torch.float32).view(1, 3, 1, 1)
_std_gpu  = torch.tensor(IMG_STD,  dtype=torch.float32).view(1, 3, 1, 1)

def to_dev(images_u8, train=False):
    """uint8 (N,3,H,W) CPU tensor -> normalised float32 CUDA tensor.
    Equivalent to torchvision ToTensor (/255) + Normalize(mean, std);
    with train=True also applies RandomHorizontalFlip(p=0.5) independently per image."""
    x = images_u8.pin_memory().to(device, non_blocking=True).float().div_(255.0)
    if train:
        flip = torch.rand(x.shape[0], device=device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    return x.sub_(_mean_gpu.to(device)).div_(_std_gpu.to(device))


class PreloadedSet:
    """All images of one split decoded + resized ONCE, kept in RAM as uint8 (N,3,H,W).
    Labels and class order from torchvision ImageFolder (identical to the baseline)."""
    def __init__(self, root, name):
        folder = ImageFolder(root)          # no transform; only used for file listing / labels
        self.classes, self.targets = folder.classes, list(folder.targets)
        paths = [pth for pth, _ in folder.samples]
        n = len(paths)
        self.images = torch.empty((n, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.uint8)
        view = self.images.numpy()          # shares memory; threads write their own slices
        def _load(i):
            with Image.open(paths[i]) as im:
                im = resize_transform(im.convert("RGB"))
            view[i] = np.asarray(im, dtype=np.uint8).transpose(2, 0, 1)
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=PRELOAD_THREADS) as pool:
            for _ in tqdm(pool.map(_load, range(n)), total=n,
                          desc=f"Preloading {name}", leave=False):
                pass
        self.labels = torch.tensor(self.targets, dtype=torch.long)
        print(f"{name}: {n} images preloaded in {time.time()-t0:.0f}s "
              f"-> {self.images.numel()/2**20:.0f} MB uint8 in RAM")
    def __len__(self): return len(self.targets)
    def get_labels(self): return self.targets   # EasyFSL TaskSampler compatibility


train_set = PreloadedSet(TRAIN_DIR, "train")
val_set   = PreloadedSet(VAL_DIR,   "val")
test_set  = PreloadedSet(TEST_DIR,  "test")
free_host_memory()
print(f"Classes: train={len(train_set.classes)}, val={len(val_set.classes)}, test={len(test_set.classes)}")
print(f"Images:  train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")
assert len(train_set.classes) >= TRAIN_WAY
assert len(val_set.classes)   >= TEST_WAY
assert len(test_set.classes)  >= TEST_WAY
print(f"[RAM] after caching datasets: {rss_gb():.2f} GB")

Preloading train:   0%|          | 0/38400 [00:00<?, ?it/s]

train: 38400 images preloaded in 18s -> 775 MB uint8 in RAM


Preloading val:   0%|          | 0/9600 [00:00<?, ?it/s]

val: 9600 images preloaded in 5s -> 194 MB uint8 in RAM


Preloading test:   0%|          | 0/12000 [00:00<?, ?it/s]

test: 12000 images preloaded in 6s -> 242 MB uint8 in RAM
Classes: train=64, val=16, test=20
Images:  train=38400, val=9600, test=12000
[RAM] after caching datasets: 1.98 GB


In [5]:
# ============================================================================
# Cell 5 — EpisodeLoader, TaskSampler patch, val/test loaders + smoke test (unchanged from master)
# ============================================================================

# EasyFSL compatibility patch (verbatim from baseline.ipynb)
def _patched_task_sampler_init(self, dataset, n_way, n_shot, n_query, n_tasks):
    Sampler.__init__(self)
    self.n_way, self.n_shot, self.n_query, self.n_tasks = n_way, n_shot, n_query, n_tasks
    self.items_per_label = defaultdict(list)
    for item, label in enumerate(dataset.get_labels()):
        self.items_per_label[int(label)].append(item)
    if len(self.items_per_label) < n_way:
        raise ValueError(
            f"Requested {n_way}-way episodes but dataset has only {len(self.items_per_label)} classes.")
    required = n_shot + n_query
    too_small = [l for l, items in self.items_per_label.items() if len(items) < required]
    if too_small:
        raise ValueError(f"Classes have fewer than {required} images: {too_small[:5]}")

TaskSampler.__init__ = _patched_task_sampler_init


class EpisodeLoader:
    """Fast collation from PreloadedSet (indices from unchanged EasyFSL TaskSampler).
    Yields (support_u8, support_labels, query_u8, query_labels, class_ids).
    Images are uint8 CPU tensors — call to_dev() for GPU preprocessing."""
    def __init__(self, store, n_way, n_tasks, shot=SHOT, query=QUERY):
        self.store   = store
        self.n_way   = n_way
        self.shot    = shot
        self.query   = query
        self.sampler = TaskSampler(store, n_way=n_way, n_shot=shot, n_query=query, n_tasks=n_tasks)
    def __len__(self): return len(self.sampler)
    def collate(self, idx):
        idx_t     = torch.as_tensor(idx, dtype=torch.long)
        lab       = self.store.labels[idx_t].tolist()
        class_ids = list({x for x in lab})
        remap     = {c: i for i, c in enumerate(class_ids)}
        labels    = torch.tensor([remap[c] for c in lab]).reshape(self.n_way, self.shot + self.query)
        imgs      = self.store.images[idx_t].reshape(
            self.n_way, self.shot + self.query, 3, IMAGE_SIZE, IMAGE_SIZE)
        return (imgs[:, :self.shot].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE),
                labels[:, :self.shot].flatten(),
                imgs[:, self.shot:].reshape(-1, 3, IMAGE_SIZE, IMAGE_SIZE),
                labels[:, self.shot:].flatten(),
                class_ids)
    def __iter__(self):
        for idx in self.sampler:
            yield self.collate(idx)


def make_loader(dataset, n_way, n_tasks, shot=SHOT, query=QUERY):
    loader = EpisodeLoader(dataset, n_way, n_tasks, shot, query)
    assert len(loader) == n_tasks, f"Loader has {len(loader)} tasks, expected {n_tasks}."
    return loader


# Fixed val and test loaders (same episodes for all configurations)
set_seed(VAL_EPISODE_SEED)
val_loader  = make_loader(val_set,  TEST_WAY, N_VALIDATION_TASKS)
set_seed(TEST_EPISODE_SEED)
test_loader = make_loader(test_set, TEST_WAY, N_TEST_TASKS)
print(f"val={len(val_loader)} tasks, test={len(test_loader)} tasks")

def check_episode(loader, way, shot=SHOT):
    s_img, s_lab, q_img, q_lab, _ = next(iter(loader))
    assert s_img.dtype == torch.uint8 and q_img.dtype == torch.uint8
    assert s_img.shape == (way * shot, 3, IMAGE_SIZE, IMAGE_SIZE), s_img.shape
    assert q_img.shape == (way * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE), q_img.shape
    assert len(s_lab) == way * shot and len(q_lab) == way * QUERY
    assert len(set(s_lab.tolist())) == way
    print(f"  {way}-way {shot}-shot episode OK: {tuple(s_img.shape)} / {tuple(q_img.shape)}")

t0 = time.time()
check_episode(val_loader,  TEST_WAY)
check_episode(test_loader, TEST_WAY)
print(f"Smoke test done in {time.time()-t0:.2f}s")
print("TaskSampler patched; EpisodeLoader ready.")

val=600 tasks, test=600 tasks
  5-way 5-shot episode OK: (25, 3, 84, 84) / (75, 3, 84, 84)
  5-way 5-shot episode OK: (25, 3, 84, 84) / (75, 3, 84, 84)
Smoke test done in 0.04s
TaskSampler patched; EpisodeLoader ready.


In [6]:
# ============================================================================
# Cell 6 — Model: Self_Attn + Task-Adaptive Dynamic Weights + PrototypicalNetworks
#   Euclidean + LayerNorm + DW (global average pooling; GeM removed).
# ============================================================================

class Self_Attn(nn.Module):
    """Self-attention module (verbatim from baseline.ipynb)."""
    def __init__(self, in_dim):
        super().__init__()
        if in_dim < 8:
            raise ValueError(f"in_dim must be >= 8, got {in_dim}")
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv   = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim,      kernel_size=1)

    def forward(self, x, gamma):
        b, c, h, w = x.shape
        n = h * w
        q = self.query_conv(x).reshape(b, -1, n).transpose(1, 2)
        k = self.key_conv(x).reshape(b, -1, n)
        attention = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value_conv(x).reshape(b, c, n)
        out = torch.bmm(v, attention.transpose(1, 2)).reshape(b, c, h, w)
        return gamma * out + x


class TaskAdaptiveWeightGenerator(nn.Module):
    """
    Dynamic Weights (DW) — verbatim from msenet-dynamicweights.ipynb.
    Computes episode-specific stage weights at both training and test time by combining:
      1. Support-Set Fisher Discriminability Ratio (inter-class prototype separation vs. intra-class variance)
      2. Query-to-Prototype Confidence Margin (normalized negative Shannon entropy across classes)
    A lightweight meta-gating network refines these 3*S task descriptors to modulate the global stage_weights.
    """
    def __init__(self, n_stages, gate_scale=0.5, fisher_temp=2.0):
        super().__init__()
        self.n_stages = n_stages
        self.gate_scale = gate_scale
        self.fisher_temp = nn.Parameter(torch.tensor(float(fisher_temp)), requires_grad=True)
        self.mix_logit = nn.Parameter(torch.tensor(0.0), requires_grad=True)  # Balances Fisher vs. Query Confidence

        hidden_dim = max(16, n_stages * 4)
        self.meta_gate = nn.Sequential(
            nn.Linear(n_stages * 3, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_stages),
        )
        # Zero-initialize final layer so dynamic modulation starts cleanly at identity (global prior)
        nn.init.zeros_(self.meta_gate[-1].weight)
        nn.init.zeros_(self.meta_gate[-1].bias)

    def forward(self, base_weights, fisher_scores, query_conf_scores, dist_means):
        # fisher_scores: (S,), query_conf_scores: (S, Q), dist_means: (S,)
        # 1. Task-level support separability distribution across stages: (S,) -> (S, 1, 1)
        fisher_norm = (fisher_scores - fisher_scores.mean()) / (fisher_scores.std(unbiased=False) + 1e-6)
        w_fisher = F.softmax(self.fisher_temp * fisher_norm, dim=0).view(self.n_stages, 1, 1) * self.n_stages

        # 2. Query-specific confidence weighting across stages: (S, Q) -> (S, Q, 1)
        conf_norm = (query_conf_scores - query_conf_scores.mean(dim=0, keepdim=True)) / (
            query_conf_scores.std(dim=0, keepdim=True, unbiased=False) + 1e-6
        )
        w_conf = F.softmax(self.fisher_temp * conf_norm, dim=0).unsqueeze(-1) * self.n_stages

        # 3. Learned neural task-context gate from episode statistics: (3*S,) -> (S, 1, 1)
        task_stats = torch.cat([
            fisher_norm,
            query_conf_scores.mean(dim=1),
            dist_means / (dist_means.mean() + 1e-6),
        ], dim=0)
        neural_delta = torch.tanh(self.meta_gate(task_stats)).view(self.n_stages, 1, 1)

        # Combine analytical Fisher/Confidence modulation with learned neural gate
        alpha = torch.sigmoid(self.mix_logit)
        stat_multiplier = alpha * w_fisher + (1.0 - alpha) * w_conf  # (S, Q, 1)

        # Anchor to positive global stage_weights (softplus prevents negative distance inversion)
        prior_weights = F.softplus(base_weights).view(self.n_stages, 1, 1)
        dynamic_weights = prior_weights * (
            (1.0 - self.gate_scale) + self.gate_scale * stat_multiplier
        ) * (1.0 + self.gate_scale * neural_delta)

        return dynamic_weights  # Shape: (S, Q, 1)


class PrototypicalNetworks(nn.Module):
    """Multi-scale prototypical network: Euclidean distance + per-stage LayerNorm +
    Task-Adaptive Dynamic Weights (always on) + global average pooling."""

    def __init__(self, backbone, image_size,
                 stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
                 weightlearnable=True, gammalearnable=True,
                 gamma1=0.2, gamma2=0.2,
                 feature_chunk_size=600, attention_chunk_size=600,
                 use_layernorm=True,
                 dynamic_gate_scale=DYNAMIC_GATE_SCALE, fisher_temp=FISHER_TEMP):
        super().__init__()
        if len(weight) != len(stage_nodes):
            raise ValueError(f"{len(weight)} weights given for {len(stage_nodes)} stages.")
        self.weightlearnable      = weightlearnable
        self.gammalearnable       = gammalearnable
        self.feature_chunk_size   = feature_chunk_size
        self.attention_chunk_size = attention_chunk_size
        self.n_stages             = len(stage_nodes)
        self.use_layernorm        = use_layernorm

        # Global stage prior weights (DW anchors to softplus of these)
        self.stage_weights = nn.Parameter(
            torch.tensor(list(weight), dtype=torch.float32), requires_grad=weightlearnable)
        self.gamma1_l = nn.Parameter(torch.tensor(float(gamma1)), requires_grad=gammalearnable)
        self.gamma2_l = nn.Parameter(torch.tensor(float(gamma2)), requires_grad=gammalearnable)

        # Build feature extractor
        graph_nodes, _ = get_graph_node_names(backbone)
        return_nodes = {}
        for i, name in enumerate(stage_nodes):
            matches = [n for n in graph_nodes if n == name or n.startswith(name + ".")]
            if not matches:
                raise ValueError(f"Stage {name!r} not found in backbone graph.")
            return_nodes[matches[-1]] = f"stage_{i}"
        print("Stage nodes:", return_nodes)
        self.feature_extractor = create_feature_extractor(backbone, return_nodes=return_nodes)

        was_training = self.feature_extractor.training
        self.feature_extractor.eval()
        with torch.no_grad():
            dummy = self.feature_extractor(torch.zeros(1, 3, image_size, image_size))
        self.feature_extractor.train(was_training)
        channels = [dummy[f"stage_{i}"].shape[1] for i in range(self.n_stages)]
        print("Selected feature channels:", channels,
              "| spatial sizes:", [tuple(dummy[f"stage_{i}"].shape[2:]) for i in range(self.n_stages)])

        self.attention_modules = nn.ModuleList([Self_Attn(c) for c in channels])
        self.global_avg_pool   = nn.AdaptiveAvgPool2d(1)

        # LayerNorm: one module per stage, shared by support and query embeddings.
        self.stage_norms = nn.ModuleList(
            [nn.LayerNorm(c) if use_layernorm else nn.Identity() for c in channels])

        # Task-Adaptive Dynamic Weighting module (created after the attention modules so the
        # attention initialisation under set_seed(SEED) is identical to the master notebook)
        self.dynamic_weight_net = TaskAdaptiveWeightGenerator(
            n_stages=self.n_stages, gate_scale=dynamic_gate_scale, fisher_temp=fisher_temp)
        self.last_dynamic_weights = None   # (S,) mean dynamic weights of the most recent episode

    def _extract_in_chunks(self, images):
        if images.size(0) <= self.feature_chunk_size:
            return self.feature_extractor(images)
        collected = None
        for chunk in images.split(self.feature_chunk_size, dim=0):
            feats = self.feature_extractor(chunk)
            if collected is None:
                collected = {k: [v] for k, v in feats.items()}
            else:
                for k, v in feats.items(): collected[k].append(v)
        return {k: torch.cat(vs, dim=0) for k, vs in collected.items()}

    def _attend_and_pool(self, features, stage_idx, gamma):
        module = self.attention_modules[stage_idx]
        if features.size(0) <= self.attention_chunk_size:
            return self.global_avg_pool(module(features, gamma)).flatten(1)
        pooled = []
        for chunk in features.split(self.attention_chunk_size, dim=0):
            pooled.append(self.global_avg_pool(module(chunk, gamma)).flatten(1))
        return torch.cat(pooled, dim=0)

    @staticmethod
    def _compute_prototypes_and_fisher(z_support, support_labels):
        """
        (Verbatim from msenet-dynamicweights.ipynb.)
        Vectorized prototype computation (zero Python loop over classes) +
        Support-Set Fisher Discriminability Ratio: (Inter-class separation) / (Intra-class spread).
        """
        unique_labels, inverse_indices, counts = torch.unique(
            support_labels, sorted=True, return_inverse=True, return_counts=True
        )
        n_way = unique_labels.size(0)
        feat_dim = z_support.size(1)

        # Vectorized class sum and mean via index_add_
        prototypes = torch.zeros(n_way, feat_dim, device=z_support.device, dtype=z_support.dtype)
        prototypes.index_add_(0, inverse_indices, z_support)
        prototypes = prototypes / counts.unsqueeze(1).to(z_support.dtype)

        # Normalize features to compute scale-invariant Fisher discriminability
        z_s_norm = F.normalize(z_support.float(), p=2, dim=-1)
        proto_norm = F.normalize(prototypes.float(), p=2, dim=-1)

        # Intra-class variance: mean cosine distance from each support sample to its class prototype
        assigned_proto = proto_norm[inverse_indices]
        intra_scatter = (1.0 - (z_s_norm * assigned_proto).sum(dim=-1)).mean()

        # Inter-class separation: mean pairwise cosine distance between distinct class prototypes
        sim_matrix = torch.mm(proto_norm, proto_norm.t())
        if n_way > 1:
            inter_sep = (1.0 - sim_matrix).sum() / (n_way * (n_way - 1))
        else:
            inter_sep = torch.tensor(1.0, device=z_support.device)

        fisher_score = inter_sep / (intra_scatter + 0.1)
        return prototypes, fisher_score.to(z_support.dtype)

    def _stage_distance(self, support_features, query_features, support_labels, stage_idx):
        z_support = self._attend_and_pool(support_features, stage_idx, self.gamma1_l)
        z_query   = self._attend_and_pool(query_features,   stage_idx, self.gamma2_l)

        # LayerNorm AFTER pool, BEFORE prototype computation
        z_support = self.stage_norms[stage_idx](z_support)
        z_query   = self.stage_norms[stage_idx](z_query)

        prototypes, fisher_score = self._compute_prototypes_and_fisher(z_support, support_labels)
        dist = torch.cdist(z_query, prototypes, p=2)          # Euclidean distance

        # Query-level confidence score: 1 - normalized entropy of stage probabilities (DW, verbatim)
        n_way = dist.size(-1)
        probs = F.softmax(-dist.float(), dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1) / np.log(max(n_way, 2))
        query_conf = (1.0 - entropy).to(dist.dtype)           # (Q,) in [0, 1], higher = sharper

        return dist, fisher_score, query_conf

    def forward(self, support_images, support_labels, query_images):
        support_feats = self._extract_in_chunks(support_images)
        query_feats   = self._extract_in_chunks(query_images)

        distances, fisher_scores, query_confs = [], [], []
        for i in range(self.n_stages):
            d, f_score, q_conf = self._stage_distance(
                support_feats[f"stage_{i}"], query_feats[f"stage_{i}"], support_labels, i)
            distances.append(d); fisher_scores.append(f_score); query_confs.append(q_conf)

        distances     = torch.stack(distances,     dim=0)   # (S, Q, way)
        fisher_scores = torch.stack(fisher_scores, dim=0)   # (S,)
        query_confs   = torch.stack(query_confs,   dim=0)   # (S, Q)
        dist_means    = distances.mean(dim=(1, 2))          # (S,)

        eff_weights = self.dynamic_weight_net(
            self.stage_weights, fisher_scores, query_confs, dist_means)   # (S, Q, 1)
        self.last_dynamic_weights = eff_weights.detach().mean(dim=(1, 2))

        return -(eff_weights * distances).sum(dim=0)

print("Model class defined.")

Model class defined.


In [7]:
# ============================================================================
# Cell 7 — ModelEMA + evaluate_full + pretrained cache + model/optimizer builders
# ============================================================================

class ModelEMA:
    """EMA of the complete model state (parameters AND BN running statistics).
    Covers every floating-point entry of the state_dict, so the DW parameters
    (fisher_temp, mix_logit, meta_gate.*) and stage_weights/gammas/LayerNorm
    are all averaged automatically. From 5-way-ablation A4, verbatim."""
    def __init__(self, model, beta):
        self.beta   = beta
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        ema_sd, live_sd = self.module.state_dict(), model.state_dict()
        fkeys = [k for k, v in ema_sd.items() if v.is_floating_point()]
        okeys = [k for k in ema_sd if k not in set(fkeys)]
        self.ema_f  = [ema_sd[k]  for k in fkeys]
        self.live_f = [live_sd[k] for k in fkeys]
        self.ema_o  = [ema_sd[k]  for k in okeys]
        self.live_o = [live_sd[k] for k in okeys]

    @torch.no_grad()
    def update(self, model=None):
        torch._foreach_mul_(self.ema_f, self.beta)
        torch._foreach_add_(self.ema_f, self.live_f, alpha=1.0 - self.beta)
        for e, l in zip(self.ema_o, self.live_o):
            e.copy_(l)


class fixed_rng:
    """Draw evaluation episodes from a fixed seed, then restore the previous RNG state."""
    def __init__(self, seed): self.seed = seed
    def __enter__(self):
        self.py = random.getstate(); self.np = np.random.get_state()
        self.t  = torch.get_rng_state()
        self.cu = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        random.seed(self.seed); np.random.seed(self.seed); torch.manual_seed(self.seed)
    def __exit__(self, *exc):
        random.setstate(self.py); np.random.set_state(self.np); torch.set_rng_state(self.t)
        if self.cu is not None: torch.cuda.set_rng_state_all(self.cu)
        return False


@torch.no_grad()
def evaluate_full(data_loader, model, seed, desc="Evaluating", timed=False):
    """Baseline evaluate + CE loss + per-episode accuracy + optional timing
    + per-episode dynamic stage weights (DW)."""
    model.eval()
    task_acc, task_loss, ep_time, dw_eps = [], [], [], []
    if timed:
        s0, l0, q0, _, _ = next(iter(data_loader))
        S0, Q0, L0 = to_dev(s0), to_dev(q0), l0.to(device)
        for _ in range(3): model(S0, L0, Q0)
        if device.type == "cuda": torch.cuda.synchronize()
    with fixed_rng(seed):
        for support_images, support_labels, query_images, query_labels, _ in \
                tqdm(data_loader, total=len(data_loader), desc=desc, leave=False):
            s_img  = to_dev(support_images)
            q_img  = to_dev(query_images)
            s_lab  = support_labels.to(device)
            labels = query_labels.to(device)
            if timed:
                if device.type == "cuda": torch.cuda.synchronize()
                t0 = time.perf_counter()
            scores = model(s_img, s_lab, q_img)
            if timed:
                if device.type == "cuda": torch.cuda.synchronize()
                ep_time.append(time.perf_counter() - t0)
            dw_eps.append(model.last_dynamic_weights.detach().float().clone())
            task_loss.append(F.cross_entropy(scores.float(), labels).item())
            task_acc.append((scores.argmax(dim=1) == labels).float().mean().item())
    if len(task_acc) != len(data_loader):
        raise RuntimeError(f"Evaluated {len(task_acc)} tasks but loader promised {len(data_loader)}.")
    task_acc = 100.0 * np.array(task_acc)
    dw_arr = torch.stack(dw_eps, dim=0).cpu().numpy()            # (n_tasks, S)
    return dict(acc=float(task_acc.mean()), loss=float(np.mean(task_loss)),
                task_acc=task_acc, per_episode_time=ep_time,
                dw_mean=dw_arr.mean(axis=0), dw_std=dw_arr.std(axis=0), dw_first=dw_arr[:2])


# Pretrained backbone cached once in RAM (all configs start from this exact state)
def _load_pretrained_state():
    if not USE_IMAGENET_PRETRAINED:
        return None
    sd = resnet18(weights=ResNet18_Weights.DEFAULT).state_dict()
    return {k: v.detach().clone() for k, v in sd.items()}

PRETRAINED_STATE = _load_pretrained_state()
print("Pretrained backbone:", "ENABLED" if PRETRAINED_STATE is not None else "DISABLED (random init)")


def build_model(key, verbose=False):
    """Build a fresh model from PRETRAINED_STATE for the given config key.
    set_seed(SEED) ensures identical random init of attention modules."""
    cfg = EXPERIMENT_CONFIGS[key]
    set_seed(SEED)
    backbone = resnet18(weights=None)
    if PRETRAINED_STATE is not None:
        backbone.load_state_dict(PRETRAINED_STATE)
    backbone.fc = nn.Identity()
    sink = contextlib.nullcontext() if verbose else contextlib.redirect_stdout(io.StringIO())
    with sink:
        model = PrototypicalNetworks(
            backbone=backbone, image_size=IMAGE_SIZE,
            stage_nodes=STAGE_NODES, weight=STAGE_WEIGHTS,
            weightlearnable=True, gammalearnable=cfg["gammalearnable"],
            gamma1=gamma_init, gamma2=gamma_init,
            feature_chunk_size=FEATURE_CHUNK_SIZE,
            attention_chunk_size=ATTENTION_CHUNK_SIZE,
            use_layernorm=cfg["use_layernorm"],
            dynamic_gate_scale=DYNAMIC_GATE_SCALE, fisher_temp=FISHER_TEMP,
        )
    return model.to(device)


def build_optimizer(model, cfg):
    """Baseline Adam param groups + DW group.
    LayerNorm params are in regular_params (backbone_lr). All DW parameters
    (fisher_temp, mix_logit, meta_gate.*) get their own group at dynamic_gate_lr."""
    dw_params = list(model.dynamic_weight_net.parameters())
    special = [model.stage_weights, model.gamma1_l, model.gamma2_l] + dw_params
    special_ids    = {id(p) for p in special}
    regular_params = [p for p in model.parameters() if id(p) not in special_ids and p.requires_grad]
    param_groups = [{"params": regular_params, "lr": backbone_lr}]
    if model.weightlearnable:
        param_groups.append({"params": [model.stage_weights], "lr": weights_lr})
    if model.gammalearnable:
        param_groups.append({"params": [model.gamma1_l, model.gamma2_l], "lr": weights_lr})
    param_groups.append({"params": dw_params, "lr": dynamic_gate_lr})
    return optim.Adam(param_groups)


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def snapshot_params(model):
    """Lightweight numerical snapshot of all tracked learnable scalars (incl. DW)."""
    d = {f"stage_w{i+1}": v for i, v in
         enumerate(model.stage_weights.detach().cpu().tolist())}
    for i, v in enumerate(F.softplus(model.stage_weights.detach()).cpu().tolist()):
        d[f"prior_w{i+1}"] = v                       # softplus(stage_weights): the DW global prior
    d["gamma1"] = float(model.gamma1_l.detach().cpu())
    d["gamma2"] = float(model.gamma2_l.detach().cpu())
    dw = model.dynamic_weight_net
    d["dw_fisher_temp"]      = float(dw.fisher_temp.detach().cpu())
    d["dw_mix_logit"]        = float(dw.mix_logit.detach().cpu())
    d["dw_mix_alpha"]        = float(torch.sigmoid(dw.mix_logit.detach()).cpu())   # Fisher share
    d["dw_gate_out_w_norm"]  = float(dw.meta_gate[-1].weight.detach().norm().cpu())
    d["dw_gate_out_b_norm"]  = float(dw.meta_gate[-1].bias.detach().norm().cpu())
    d["dw_gate_total_norm"]  = float(torch.sqrt(sum((p.detach().float() ** 2).sum()
                                                    for p in dw.parameters())).cpu())
    return d


def load_checkpoint_model(path):
    """Rebuild the complete model from a saved .pth (architecture config + weights).
    Independent of PRETRAINED_STATE: the backbone weights come from the checkpoint itself."""
    ckpt = torch.load(path, map_location="cpu")
    mc = ckpt["model_config"]
    backbone = resnet18(weights=None)
    backbone.fc = nn.Identity()
    with contextlib.redirect_stdout(io.StringIO()):
        model = PrototypicalNetworks(
            backbone=backbone, image_size=mc["image_size"],
            stage_nodes=tuple(mc["stage_nodes"]), weight=tuple(mc["stage_weights_init"]),
            weightlearnable=mc["weightlearnable"], gammalearnable=mc["gammalearnable"],
            gamma1=mc["gamma_init"], gamma2=mc["gamma_init"],
            feature_chunk_size=mc["feature_chunk_size"], attention_chunk_size=mc["attention_chunk_size"],
            use_layernorm=mc["use_layernorm"],
            dynamic_gate_scale=mc["dynamic_gate_scale"], fisher_temp=mc["fisher_temp_init"],
        )
    model.load_state_dict(ckpt["model_state_dict"])
    return model.to(device).eval(), ckpt


print("Helpers defined.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 215MB/s]

Pretrained backbone: ENABLED
Helpers defined.


In [8]:
# ============================================================================
# Cell 8 — Smoke test: forward/backward, DW updates, EMA updates, LayerNorm presence,
#   and a save/load round-trip of the checkpoint format
# ============================================================================
print("Running smoke test...")
t0 = time.time()

DW_KEYS_REQUIRED = ["fisher_temp", "mix_logit", "meta_gate.3.weight", "meta_gate.3.bias"]

key, cfg = KEY, EXPERIMENT_CONFIGS[KEY]
m = build_model(key)
n_total, n_trainable = count_params(m)
named_dw = list(m.dynamic_weight_net.named_parameters())
dw_ids   = {id(p) for _, p in named_dw}

# ── structural checks ──────────────────────────────────────────────────────
assert not hasattr(m, "gem_raw"), "GeM must not exist in this model"
assert not hasattr(m, "cosine_temp"), "cosine temperature must not exist"
assert len(m.stage_norms) == 5 and all(isinstance(ln, nn.LayerNorm) for ln in m.stage_norms), \
    "expected one LayerNorm per stage"
assert m.gammalearnable and cfg["use_ema"]
assert all(k in dict(named_dw) for k in DW_KEYS_REQUIRED)
assert float(m.dynamic_weight_net.fisher_temp) == FISHER_TEMP
assert float(m.dynamic_weight_net.mix_logit) == 0.0
assert float(m.dynamic_weight_net.meta_gate[-1].weight.abs().sum()) == 0.0
assert float(m.dynamic_weight_net.meta_gate[-1].bias.abs().sum()) == 0.0
assert m.dynamic_weight_net.gate_scale == DYNAMIC_GATE_SCALE

# ── optimizer covers every trainable parameter exactly once; DW group lr ──
opt = build_optimizer(m, cfg)
grouped = [id(p) for g in opt.param_groups for p in g["params"]]
assert len(grouped) == len(set(grouped)), "parameter appears in more than one optimizer group"
assert set(grouped) == {id(p) for p in m.parameters() if p.requires_grad}, "optimizer misses parameters"
dw_groups = [g for g in opt.param_groups if any(id(p) in dw_ids for p in g["params"])]
assert len(dw_groups) == 1 and {id(p) for p in dw_groups[0]["params"]} == dw_ids
assert dw_groups[0]["lr"] == dynamic_gate_lr

s  = torch.randn(TEST_WAY * SHOT,  3, IMAGE_SIZE, IMAGE_SIZE, device=device)
sl = torch.arange(TEST_WAY, device=device).repeat_interleave(SHOT)
q  = torch.randn(TEST_WAY * QUERY, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
ql = torch.arange(TEST_WAY, device=device).repeat_interleave(QUERY)

# ── AMP forward + backward (the dtype path used in training) ──────────────
m.train()
if AMP_ON:
    with torch.amp.autocast("cuda", enabled=True):
        logits = m(s, sl, q)
        loss = F.cross_entropy(logits.float(), ql)
    loss.backward()
    assert logits.shape == (TEST_WAY * QUERY, TEST_WAY), logits.shape
    assert torch.isfinite(loss), "non-finite AMP loss"
    opt.zero_grad(set_to_none=True)

# ── fp32 optimisation steps: DW grads + DW updates + EMA tracking ──────────
ema = ModelEMA(m, EMA_BETA)
ema_sd = ema.module.state_dict()
dw_state_keys = [f"dynamic_weight_net.{n}" for n, _ in named_dw]
assert all(k in ema_sd and ema_sd[k].is_floating_point() for k in dw_state_keys), "DW parameters missing from EMA state"

before = {n: p.detach().clone() for n, p in named_dw}
stage_w_before = m.stage_weights.detach().clone()
gamma1_before  = m.gamma1_l.detach().clone()
ema_checked = 0
for step in range(3):
    opt.zero_grad(set_to_none=True)
    logits = m(s, sl, q)
    assert logits.shape == (TEST_WAY * QUERY, TEST_WAY), logits.shape
    loss = F.cross_entropy(logits, ql)
    assert torch.isfinite(loss)
    loss.backward()
    pdict = dict(named_dw)
    for k in DW_KEYS_REQUIRED:
        g = pdict[k].grad
        assert g is not None and torch.isfinite(g).all() and g.abs().sum() > 0, f"no/zero grad for DW {k} (step {step})"
    assert m.stage_weights.grad is not None and m.stage_weights.grad.abs().sum() > 0
    assert m.gamma1_l.grad is not None and torch.isfinite(m.gamma1_l.grad).all()
    lw = m.last_dynamic_weights
    assert lw is not None and lw.shape == (5,) and torch.isfinite(lw).all() and (lw > 0).all()
    torch.nn.utils.clip_grad_norm_([p for p in m.parameters() if p.requires_grad], max_norm=5.0)
    prev_ema = {k: ema_sd[k].clone() for k in dw_state_keys}
    opt.step()
    ema.update()
    live_sd = m.state_dict()
    for k in dw_state_keys:                       # EMA == beta*prev + (1-beta)*live
        expect = EMA_BETA * prev_ema[k] + (1.0 - EMA_BETA) * live_sd[k]
        assert torch.allclose(ema_sd[k], expect, atol=1e-6, rtol=1e-5), f"EMA mismatch on {k}"
        ema_checked += 1

for n in ("fisher_temp", "mix_logit", "meta_gate.3.weight"):    # DW parameters were optimised
    assert not torch.equal(before[n], dict(named_dw)[n].detach()), f"DW param {n} did not update"
assert not torch.equal(stage_w_before, m.stage_weights.detach()), "stage_weights did not update"
assert not torch.equal(gamma1_before, m.gamma1_l.detach()), "gamma1 did not update"
assert not torch.equal(ema_sd["dynamic_weight_net.meta_gate.3.weight"],
                       m.state_dict()["dynamic_weight_net.meta_gate.3.weight"]), "EMA should lag the live weights"

# ── eval-mode forward with the EMA model ──────────────────────────────────
with torch.no_grad():
    ema.module.eval()
    out = ema.module(s, sl, q)
assert out.shape == (TEST_WAY * QUERY, TEST_WAY) and torch.isfinite(out).all()
assert ema.module.last_dynamic_weights.shape == (5,)

# ── checkpoint save / load round-trip (same format as the final .pth) ─────
_tmp = MODEL_DIR / "_smoke_roundtrip.pth"
torch.save({"model_state_dict": {k: v.detach().cpu() for k, v in ema.module.state_dict().items()},
            "model_config": MODEL_CONFIG}, _tmp)
m2, _ = load_checkpoint_model(_tmp)
with torch.no_grad():
    out2 = m2(s, sl, q)
assert torch.allclose(out, out2, atol=1e-4, rtol=1e-4), "checkpoint round-trip changed the outputs"
_tmp.unlink()

snap = snapshot_params(m)
for k in ("dw_fisher_temp", "dw_mix_alpha", "dw_gate_out_w_norm", "prior_w1", "stage_w5"):
    assert k in snap and math.isfinite(snap[k])
assert not any(k.startswith("gem_") for k in snap)

print(f"  [{key}] {cfg['description']:25s} total={n_total:,}  trainable={n_trainable:,}  "
      f"DW params={sum(p.numel() for _, p in named_dw):,}  EMA DW checks={ema_checked}  "
      f"checkpoint round-trip OK")
del m, m2, opt, ema
free_gpu()
print(f"Smoke test passed in {time.time()-t0:.1f}s")

Running smoke test...
  [DW_LN_EMA] DW + LayerNorm + EMA      total=11,620,634  trainable=11,620,634  DW params=467  EMA DW checks=24  checkpoint round-trip OK
Smoke test passed in 7.1s


In [9]:
# ============================================================================
# Cell 9 — run_experiment(key): complete training + evaluation of the single configuration
#   (Euclidean + Dynamic Weights + LayerNorm + EMA, no GeM). Validation/test use the EMA
#   weights; the best-validation EMA model is saved completely to a .pth file.
# ============================================================================

RESULTS = {}   # filled by Cell 10

def run_experiment(key):
    cfg = EXPERIMENT_CONFIGS[key]
    use_ema = cfg["use_ema"]
    print("=" * 72)
    print(f"  CONFIG [{key}]: {cfg['description']}")
    print(f"  Euclidean + DynamicWeights  |  LayerNorm={cfg['use_layernorm']}  "
          f"|  EMA={use_ema}  |  GeM=False (global avg pooling)  |  gammalearnable={cfg['gammalearnable']}")
    print("=" * 72)
    free_gpu()
    wall0 = time.time()

    # ── Build fresh model + optimizer ──────────────────────────────────────
    model     = build_model(key, verbose=True)
    optimizer = build_optimizer(model, cfg)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda", enabled=AMP_ON)
    trainable = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss()
    n_total, n_trainable = count_params(model)
    print(f"  Parameters: total={n_total:,}  trainable={n_trainable:,}")

    # ── EMA: wraps the model (incl. DW parameters); validation/test use EMA weights ──
    if use_ema:
        ema        = ModelEMA(model, EMA_BETA)
        eval_model = ema.module
    else:
        ema        = None
        eval_model = model

    trajectory = [dict(step=0, epoch=0, **snapshot_params(model))]
    best_val_acc, best_epoch, best_state = -1.0, 0, None
    epoch_log        = []
    train_time_total = 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # ── Epoch loop ──────────────────────────────────────────────────────────
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        # All configs see identical training episodes (seeded per-epoch)
        set_seed(TRAIN_EPISODE_SEED + (epoch - 1))
        train_loader_epoch = make_loader(train_set, TRAIN_WAY, N_TRAINING_EPISODES)

        loss_sum   = torch.zeros((), dtype=torch.float64, device=device)
        acc_sum    = loss_sum.clone()
        n_episodes = 0
        lr_now     = optimizer.param_groups[0]["lr"]

        if device.type == "cuda": torch.cuda.synchronize()
        t_train0 = time.time()

        progress = tqdm(train_loader_epoch, total=len(train_loader_epoch),
                        desc=f"[{key}] Epoch {epoch}/{MAX_EPOCHS}", leave=False)

        for support_images, support_labels, query_images, query_labels, _ in progress:
            optimizer.zero_grad(set_to_none=True)
            sl   = support_labels.to(device, non_blocking=True)
            ql   = query_labels.to(device,   non_blocking=True)
            S_in = to_dev(support_images, train=True)
            Q_in = to_dev(query_images,   train=True)

            with torch.amp.autocast("cuda", enabled=AMP_ON):
                scores = model(S_in, sl, Q_in)
                ce     = criterion(scores, ql)

            scaler.scale(ce).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            if use_ema:
                ema.update()

            loss_sum += ce.detach().double()
            acc_sum  += (scores.detach().argmax(dim=1) == ql).float().mean().double()
            n_episodes += 1

        if device.type == "cuda": torch.cuda.synchronize()
        train_time = time.time() - t_train0
        train_time_total += train_time

        if n_episodes != N_TRAINING_EPISODES:
            raise RuntimeError(f"Ran {n_episodes} episodes, expected {N_TRAINING_EPISODES}.")

        scheduler.step()
        train_loss = float(loss_sum.item() / n_episodes)
        train_acc  = 100.0 * float(acc_sum.item() / n_episodes)

        # Snapshot learnable params (live model) after this epoch
        trajectory.append(dict(
            step=epoch * N_TRAINING_EPISODES, epoch=epoch, **snapshot_params(model)))

        # ── Validation (EMA weights) ───────────────────────────────────────
        t_val0 = time.time()
        val    = evaluate_full(val_loader, eval_model, VAL_EPISODE_SEED,
                               desc=f"[{key}] Val epoch {epoch}")
        val_time = time.time() - t_val0

        row = dict(
            config=key, epoch=epoch,
            train_acc=train_acc, train_loss=train_loss,
            val_acc=val["acc"],  val_loss=val["loss"],
            train_time_s=train_time, val_time_s=val_time, lr=lr_now,
            gpu_peak_mb=(torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0),
            **snapshot_params(model),
        )
        # DW tracking: mean/std dynamic stage weights over the val episodes, plus the EMA-model DW scalars.
        if use_ema:
            row.update({f"ema_{k}": v for k, v in snapshot_params(eval_model).items() if k.startswith("dw_")})
        for i in range(5):
            row[f"dw_val_w{i+1}"]     = float(val["dw_mean"][i])
            row[f"dw_val_w{i+1}_std"] = float(val["dw_std"][i])
        epoch_log.append(row)

        # ── Epoch print (exact required format) ────────────────────────────
        print(f"Epoch {epoch}: train loss(CE)={train_loss:.4f}, train acc={train_acc:.2f}%, "
              f"val loss={val['loss']:.4f}, val accuracy={val['acc']:.2f}%, "
              f"time={(train_time+val_time)/60:.1f} min, "
              f"RAM={ram_guard(f'{key} epoch {epoch}'):.1f} GB")
        dw_temp_src  = row.get("ema_dw_fisher_temp", row["dw_fisher_temp"])
        dw_alpha_src = row.get("ema_dw_mix_alpha",   row["dw_mix_alpha"])
        print(f"          DW val mean stage weights={[round(float(v), 4) for v in val['dw_mean']]}, "
              f"fisher_temp={dw_temp_src:.4f}, fisher mix alpha={dw_alpha_src:.4f}")

        if val["acc"] > best_val_acc:
            best_val_acc, best_epoch = val["acc"], epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in eval_model.state_dict().items()}

    peak_train_gpu_mb = (torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0)
    print(f"[{key}] Best validation: {best_val_acc:.2f}% @ epoch {best_epoch}")

    # ── Release training objects; rebuild fresh model for test ────────────
    params_end = snapshot_params(model)
    del model, optimizer, scheduler, scaler, eval_model, trainable
    if ema is not None:
        del ema
    del train_loader_epoch, progress, loss_sum, acc_sum
    free_gpu()

    infer_model = build_model(key)
    infer_model.load_state_dict(best_state)
    del best_state
    free_gpu()  # resets peak stats so inference memory is measured separately

    # ── Final test (5-way 5-shot, timed) ───────────────────────────────────
    test = evaluate_full(test_loader, infer_model, TEST_EPISODE_SEED,
                         desc=f"[{key}] Test 5-way 5-shot", timed=True)
    peak_infer_gpu_mb = (torch.cuda.max_memory_allocated()/2**20
                         if device.type == "cuda" else 0.0)
    params_best = snapshot_params(infer_model)

    t_arr = np.array(test["per_episode_time"]) * 1000.0
    infer_ms_mean = float(t_arr.mean()) if len(t_arr) > 0 else float("nan")

    # ── Required final output format ────────────────────────────────────────
    print("=" * 48)
    print(f"[{key}] FINAL TEST ACCURACY 5-way 5-shot: {test['acc']:.2f}%")
    print("=" * 48)
    sw = [round(params_best[f"stage_w{i+1}"], 4) for i in range(5)]
    g1 = round(params_best["gamma1"], 4)
    g2 = round(params_best["gamma2"], 4)
    print(f"Stage weights (tested model): {sw}")
    print(f"Learned global prior scale weights (softplus): "
          f"{[round(params_best[f'prior_w{i+1}'], 4) for i in range(5)]}")
    print(f"gamma1/gamma2 (tested model): {g1} {g2}  (learnable={cfg['gammalearnable']})")
    print(f"DW fisher_temp={params_best['dw_fisher_temp']:.4f}  "
          f"fisher_vs_conf_mix (sigmoid)={params_best['dw_mix_alpha']:.4f}")
    print(f"Test-time dynamic weights (mean across {N_TEST_TASKS} tasks): "
          f"{[round(float(x), 4) for x in test['dw_mean']]}")
    print(f"Test-time dynamic weights (std across {N_TEST_TASKS} tasks):  "
          f"{[round(float(x), 4) for x in test['dw_std']]}")
    print(f"Sample Task #1 dynamic weights: {[round(float(x), 4) for x in test['dw_first'][0]]}")
    print(f"Sample Task #2 dynamic weights: {[round(float(x), 4) for x in test['dw_first'][1]]}")
    print(f"[RAM] {key} done: {rss_gb():.2f} GB")

    test_std = float(np.array(test["task_acc"]).std(ddof=1))
    test_ci  = float(1.96 * test_std / math.sqrt(len(test["task_acc"])))
    print(f"FINAL TEST ACCURACY {TEST_WAY}-way {SHOT}-shot: {test['acc']:.2f}% ± {test_ci:.2f} (95% CI over {N_TEST_TASKS} episodes)"
          f"   |   best val {best_val_acc:.2f}% @ epoch {best_epoch}")

    # ── Save the COMPLETE model (EMA weights of the best-validation epoch, i.e. exactly the tested model)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": {k: v.detach().cpu() for k, v in infer_model.state_dict().items()},
        "model_config": MODEL_CONFIG,
        "experiment_config": dict(cfg),
        "best_epoch": int(best_epoch), "best_val_acc": float(best_val_acc),
        "test_acc": float(test["acc"]), "test_acc_ci95": test_ci, "test_loss": float(test["loss"]),
        "n_test_tasks": int(N_TEST_TASKS), "protocol": f"{TEST_WAY}-way {SHOT}-shot",
        "learned_params": params_best,
        "dw_test_mean": [float(x) for x in test["dw_mean"]],
        "dw_test_std":  [float(x) for x in test["dw_std"]],
        "hyperparameters": dict(max_epochs=MAX_EPOCHS, n_training_episodes=N_TRAINING_EPISODES,
                                train_way=TRAIN_WAY, test_way=TEST_WAY, shot=SHOT, query=QUERY,
                                backbone_lr=backbone_lr, weights_lr=weights_lr,
                                dynamic_gate_lr=dynamic_gate_lr, ema_beta=EMA_BETA, seed=SEED),
        "env": {k: str(v) for k, v in ENV_INFO.items()},
    }, CKPT_PATH)
    print(f"Saved complete model: {CKPT_PATH}  ({CKPT_PATH.stat().st_size/1e6:.1f} MB)")

    del infer_model
    free_gpu()

    return dict(
        key=key, cfg=cfg,
        epoch_log=epoch_log, trajectory=trajectory,
        params_end_of_training=params_end, params_best_checkpoint=params_best,
        best_epoch=best_epoch, best_val_acc=best_val_acc,
        test_acc=test["acc"],  test_loss=test["loss"],
        per_episode_acc=test["task_acc"].tolist(),
        dw_test_mean=test["dw_mean"].tolist(), dw_test_std=test["dw_std"].tolist(),
        dw_test_first=test["dw_first"].tolist(),
        train_time_total_s=train_time_total,
        peak_train_gpu_mb=peak_train_gpu_mb,
        peak_infer_gpu_mb=peak_infer_gpu_mb,
        infer_ms_per_episode=infer_ms_mean,
        n_total=n_total, n_trainable=n_trainable,
        wall_time_s=time.time() - wall0,
        use_ema=use_ema, test_acc_ci95=test_ci,
    )

In [10]:
# ============================================================================
# Cell 10 — Run the single experiment, then re-load the saved .pth and re-test it
# ============================================================================
RESULTS = {}
t_all = time.time()

RESULTS[KEY] = run_experiment(KEY)
free_gpu()
ram_guard(f"after {KEY}")
print(f"\nExperiment finished in {(time.time()-t_all)/60:.1f} min\n")

r = RESULTS[KEY]
print(f"  [{KEY}] {EXPERIMENT_CONFIGS[KEY]['description']:25s}  "
      f"test={r['test_acc']:.2f}%  best_val={r['best_val_acc']:.2f}%  epoch={r['best_epoch']}")

# ── Verify the .pth is a complete, self-contained model: rebuild from file and re-test ──
reloaded, ckpt = load_checkpoint_model(CKPT_PATH)
re_test = evaluate_full(test_loader, reloaded, TEST_EPISODE_SEED, desc="Re-test from .pth")
print("=" * 56)
print(f"TEST ACCURACY ({TEST_WAY}-way {SHOT}-shot)  trained model : {r['test_acc']:.2f}%")
print(f"TEST ACCURACY ({TEST_WAY}-way {SHOT}-shot)  reloaded .pth : {re_test['acc']:.2f}%")
print("=" * 56)
assert abs(re_test["acc"] - r["test_acc"]) < 0.05, "reloaded .pth does not reproduce the test accuracy"
print(f"Reloaded checkpoint reproduces the test accuracy.  File: {CKPT_PATH.name}")
del reloaded, ckpt
free_gpu()

  CONFIG [DW_LN_EMA]: DW + LayerNorm + EMA
  Euclidean + DynamicWeights  |  LayerNorm=True  |  EMA=True  |  GeM=False (global avg pooling)  |  gammalearnable=True
Stage nodes: {'maxpool': 'stage_0', 'layer1.1.relu_1': 'stage_1', 'layer2.1.relu_1': 'stage_2', 'layer3.1.relu_1': 'stage_3', 'layer4.1.relu_1': 'stage_4'}
Selected feature channels: [64, 64, 128, 256, 512] | spatial sizes: [(21, 21), (21, 21), (11, 11), (6, 6), (3, 3)]
  Parameters: total=11,620,634  trainable=11,620,634


[DW_LN_EMA] Epoch 1/10:   0%|          | 0/360 [00:00<?, ?it/s]

[DW_LN_EMA] Val epoch 1:   0%|          | 0/600 [00:00<?, ?it/s]

Epoch 1: train loss(CE)=0.8457, train acc=79.06%, val loss=0.4326, val accuracy=84.26%, time=2.4 min, RAM=3.8 GB
          DW val mean stage weights=[0.3057, 0.3135, 0.3346, 0.437, 2.003], fisher_temp=1.9265, fisher mix alpha=0.5211


[DW_LN_EMA] Epoch 2/10:   0%|          | 0/360 [00:00<?, ?it/s]

[DW_LN_EMA] Val epoch 2:   0%|          | 0/600 [00:00<?, ?it/s]

... (25 outputs left out)

[DW_LN_EMA] Test 5-way 5-shot:   0%|          | 0/600 [00:00<?, ?it/s]

[DW_LN_EMA] FINAL TEST ACCURACY 5-way 5-shot: 85.33%
Stage weights (tested model): [0.6065, 0.6791, 0.7362, 0.8114, 0.9524]
Learned global prior scale weights (softplus): [1.0417, 1.0893, 1.1275, 1.1789, 1.2787]
gamma1/gamma2 (tested model): 0.7374 0.099  (learnable=True)
DW fisher_temp=1.9265  fisher_vs_conf_mix (sigmoid)=0.5211
Test-time dynamic weights (mean across 600 tasks): [0.3079, 0.3135, 0.3386, 0.4389, 1.9956]
Test-time dynamic weights (std across 600 tasks):  [0.0071, 0.0009, 0.0107, 0.0343, 0.0412]
Sample Task #1 dynamic weights: [0.3037, 0.3143, 0.3394, 0.4537, 1.9828]
Sample Task #2 dynamic weights: [0.3059, 0.3125, 0.3653, 0.4358, 1.9683]
[RAM] DW_LN_EMA done: 3.87 GB
FINAL TEST ACCURACY 5-way 5-shot: 85.33% ± 0.49 (95% CI over 600 episodes)   |   best val 84.26% @ epoch 1
Saved complete model: /kaggle/working/results/model/msenet_dw_ln_ema_5way5shot_best.pth  (46.6 MB)

Experiment finished in 25.5 min

  [DW_LN_EMA] DW + LayerNorm + EMA       test=85.33%  best_val=84.26

Re-test from .pth:   0%|          | 0/600 [00:00<?, ?it/s]

TEST ACCURACY (5-way 5-shot)  trained model : 85.33%
TEST ACCURACY (5-way 5-shot)  reloaded .pth : 85.33%
Reloaded checkpoint reproduces the test accuracy.  File: msenet_dw_ln_ema_5way5shot_best.pth


In [11]:
# ============================================================================
# Cell 11 — CSV outputs (single configuration)
# ============================================================================
r   = RESULTS[KEY]
cfg = EXPERIMENT_CONFIGS[KEY]
ep  = np.array(r["per_episode_acc"])
std = float(ep.std(ddof=1))
ci  = float(1.96 * std / math.sqrt(len(ep)))

# 1. Test summary (one row)
df_main = pd.DataFrame([dict(
    experiment=KEY, description=cfg["description"], distance_metric="euclidean",
    use_layernorm=cfg["use_layernorm"], use_ema=cfg["use_ema"], use_gem=False,
    test_acc_5way5shot=r["test_acc"], test_acc_ci95=ci, test_acc_std=std, test_loss=r["test_loss"],
    best_val_acc=r["best_val_acc"], final_val_acc=r["epoch_log"][-1]["val_acc"],
    final_val_loss=r["epoch_log"][-1]["val_loss"], best_epoch=r["best_epoch"],
    train_time_min=r["train_time_total_s"] / 60, wall_time_min=r["wall_time_s"] / 60,
    n_total_params=r["n_total"], n_trainable_params=r["n_trainable"],
    peak_train_gpu_mb=r["peak_train_gpu_mb"], peak_infer_gpu_mb=r["peak_infer_gpu_mb"],
    infer_ms_per_episode=r["infer_ms_per_episode"],
)])
df_main.to_csv(COMP_DIR / "test_summary.csv", index=False, float_format="%.6g")
print("Saved: test_summary.csv")

# 2. Per-epoch metrics (live + EMA DW parameters and mean/std dynamic weights on val)
pd.DataFrame(r["epoch_log"]).to_csv(COMP_DIR / "per_epoch_metrics.csv", index=False, float_format="%.6g")
print("Saved: per_epoch_metrics.csv")

# 3. Per-episode test accuracy
pd.DataFrame(dict(episode=np.arange(1, len(ep) + 1), accuracy=ep)).to_csv(
    COMP_DIR / "per_episode_test_accuracy.csv", index=False, float_format="%.6g")
print("Saved: per_episode_test_accuracy.csv")

# 4. Dynamic-weight statistics on the test episodes (tested = best-EMA checkpoint)
dw_rows = []
for i in range(5):
    dw_rows.append(dict(
        stage=i + 1, stage_node=STAGE_NODES[i],
        global_prior_softplus=r["params_best_checkpoint"][f"prior_w{i+1}"],
        dyn_weight_mean_test=r["dw_test_mean"][i], dyn_weight_std_test=r["dw_test_std"][i],
        dyn_weight_task1=r["dw_test_first"][0][i], dyn_weight_task2=r["dw_test_first"][1][i],
    ))
pd.DataFrame(dw_rows).to_csv(COMP_DIR / "dynamic_weights_test_stats.csv", index=False, float_format="%.6g")
print("Saved: dynamic_weights_test_stats.csv")

# 5. Computational efficiency
pd.DataFrame([dict(
    experiment=KEY, train_time_total_s=r["train_time_total_s"],
    train_time_per_epoch_s=r["train_time_total_s"] / MAX_EPOCHS,
    infer_ms_per_episode=r["infer_ms_per_episode"],
    n_total_params=r["n_total"], n_trainable_params=r["n_trainable"],
    peak_train_gpu_mb=r["peak_train_gpu_mb"], peak_infer_gpu_mb=r["peak_infer_gpu_mb"],
    process_rss_gb=rss_gb(),
)]).to_csv(COMP_DIR / "computational_efficiency.csv", index=False, float_format="%.6g")
print("Saved: computational_efficiency.csv")

# 6a. Parameter trajectories
pd.DataFrame([dict(experiment=KEY, **snap) for snap in r["trajectory"]]).to_csv(
    PARAM_DIR / "parameter_trajectories.csv", index=False, float_format="%.6g")
print("Saved: parameter_trajectories.csv")

# 6b. Final parameters
pd.DataFrame([
    dict(experiment=KEY, source="best_val_checkpoint", **r["params_best_checkpoint"]),
    dict(experiment=KEY, source="end_of_training_ep10", **r["params_end_of_training"]),
]).to_csv(PARAM_DIR / "final_parameters.csv", index=False, float_format="%.6g")
print("Saved: final_parameters.csv")

# 7. Experiment config
config_rows = [
    {"key": "distance_metric", "value": "euclidean"},
    {"key": "seed", "value": SEED},
    {"key": "train_episode_seed", "value": TRAIN_EPISODE_SEED},
    {"key": "val_episode_seed",   "value": VAL_EPISODE_SEED},
    {"key": "test_episode_seed",  "value": TEST_EPISODE_SEED},
    {"key": "max_epochs",         "value": MAX_EPOCHS},
    {"key": "n_training_episodes","value": N_TRAINING_EPISODES},
    {"key": "n_val_tasks",        "value": N_VALIDATION_TASKS},
    {"key": "n_test_tasks",       "value": N_TEST_TASKS},
    {"key": "train_way",          "value": TRAIN_WAY},
    {"key": "test_way",           "value": TEST_WAY},
    {"key": "shot",               "value": SHOT},
    {"key": "query",              "value": QUERY},
    {"key": "image_size",         "value": IMAGE_SIZE},
    {"key": "backbone_lr",        "value": backbone_lr},
    {"key": "weights_lr",         "value": weights_lr},
    {"key": "dynamic_gate_lr",    "value": dynamic_gate_lr},
    {"key": "dynamic_gate_scale", "value": DYNAMIC_GATE_SCALE},
    {"key": "fisher_temp_init",   "value": FISHER_TEMP},
    {"key": "ema_beta",           "value": EMA_BETA},
    {"key": "use_amp",            "value": USE_AMP},
    {"key": "stage_weights_init", "value": str(STAGE_WEIGHTS)},
    {"key": "gamma_init",         "value": gamma_init},
    {"key": "use_imagenet_pretrained", "value": USE_IMAGENET_PRETRAINED},
    {"key": "checkpoint",         "value": CKPT_PATH.name},
] + [{"key": f"env_{k}", "value": str(v)} for k, v in ENV_INFO.items()] \
  + [{"key": f"config_{k}", "value": str(v)} for k, v in EXPERIMENT_CONFIGS.items()]
pd.DataFrame(config_rows).to_csv(COMP_DIR / "experiment_config.csv", index=False)
print("Saved: experiment_config.csv")
print("\nAll 8 CSV files saved.")

Saved: test_summary.csv
Saved: per_epoch_metrics.csv
Saved: per_episode_test_accuracy.csv
Saved: dynamic_weights_test_stats.csv
Saved: computational_efficiency.csv
Saved: parameter_trajectories.csv
Saved: final_parameters.csv
Saved: experiment_config.csv

All 8 CSV files saved.


In [12]:
# ============================================================================
# Cell 12 — Plots (high-resolution PNG, single configuration)
# ============================================================================
r      = RESULTS[KEY]
LABEL  = f"[{KEY}] {EXPERIMENT_CONFIGS[KEY]['description']}"
C_MAIN, C_ALT = "#4c78a8", "#f58518"
EPOCHS = list(range(1, MAX_EPOCHS + 1))
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

def _save(fig, fname):
    fig.tight_layout()
    fig.savefig(PLOT_DIR / fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {fname}")

def line_plot(cols, labels, title, ylabel, fname, mark_best=False):
    fig, ax = plt.subplots(figsize=(9, 5))
    for col, lab, colr in zip(cols, labels, (C_MAIN, C_ALT)):
        ax.plot(EPOCHS, [row[col] for row in r["epoch_log"]], marker="o", ms=4, lw=1.8, label=lab, color=colr)
    if mark_best:
        ax.axvline(r["best_epoch"], ls=":", color="gray")
        ax.annotate(f"best epoch {r['best_epoch']}\n({r['best_val_acc']:.2f}%)",
                    (r["best_epoch"], r["best_val_acc"]), textcoords="offset points",
                    xytext=(8, -28), fontsize=9)
    ax.set_title(f"{title}\n{LABEL}"); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_xticks(EPOCHS); ax.legend(fontsize=9)
    _save(fig, fname)

stage_colors = plt.cm.viridis(np.linspace(0, 0.9, 5))
ep_acc  = np.array(r["per_episode_acc"])
test_ci = 1.96 * ep_acc.std(ddof=1) / math.sqrt(len(ep_acc))

# 01 Test accuracy (the headline result)
fig, ax = plt.subplots(figsize=(7, 5.5))
names = ["Best validation", "Final-epoch validation", f"TEST ({TEST_WAY}-way {SHOT}-shot)"]
vals  = [r["best_val_acc"], r["epoch_log"][-1]["val_acc"], r["test_acc"]]
errs  = [0, 0, test_ci]
bars  = ax.bar(names, vals, yerr=errs, capsize=6, color=["#9ecae1", "#9ecae1", C_MAIN])
ax.bar_label(bars, fmt="%.2f%%", padding=4, fontsize=11, fontweight="bold")
lo, hi = min(vals), max(vals)
ax.set_ylim(lo - max(3, 2 * (hi - lo)), hi + max(2, hi - lo))
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"Test Accuracy = {r['test_acc']:.2f}%  (95% CI ±{test_ci:.2f}, {N_TEST_TASKS} episodes)\n{LABEL}")
ax.grid(axis="x", visible=False)
_save(fig, "01_test_accuracy.png")

# 02 Per-episode test accuracy distribution
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ep_acc, bins=25, color=C_MAIN, alpha=0.75, edgecolor="white")
ax.axvline(r["test_acc"], color="crimson", lw=2, label=f"mean = {r['test_acc']:.2f}%")
ax.axvspan(r["test_acc"] - test_ci, r["test_acc"] + test_ci, color="crimson", alpha=0.15, label=f"95% CI ±{test_ci:.2f}")
ax.set_xlabel("Episode accuracy (%)"); ax.set_ylabel(f"# of the {N_TEST_TASKS} test episodes")
ax.set_title(f"Per-Episode Test Accuracy ({DIST_LABEL}, {TEST_WAY}-way {SHOT}-shot)\n{LABEL}")
ax.legend()
_save(fig, "02_per_episode_test_accuracy.png")

# 03 Training vs validation accuracy
line_plot(["train_acc", "val_acc"], ["Train (30-way)", f"Validation ({TEST_WAY}-way)"],
          f"Accuracy vs Epoch ({DIST_LABEL})", "Accuracy (%)", "03_accuracy_vs_epoch.png", mark_best=False)

# 04 Training vs validation loss
line_plot(["train_loss", "val_loss"], ["Train", "Validation"],
          f"Cross-Entropy Loss vs Epoch ({DIST_LABEL})", "Cross-Entropy Loss", "04_loss_vs_epoch.png")

# 05 Validation accuracy with best epoch marked
line_plot(["val_acc"], [f"Validation ({TEST_WAY}-way {SHOT}-shot)"],
          f"Validation Accuracy vs Epoch ({DIST_LABEL})", "Accuracy (%)", "05_val_accuracy_vs_epoch.png", mark_best=True)

# 06 Global stage-weight parameter trajectories
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(5):
    vals_ = [r["trajectory"][0][f"stage_w{i+1}"]] + [row[f"stage_w{i+1}"] for row in r["epoch_log"]]
    ax.plot(range(0, MAX_EPOCHS + 1), vals_, marker="o", ms=3, color=stage_colors[i], label=f"w{i+1} ({STAGE_NODES[i]})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Global stage weight (raw; DW applies softplus)"); ax.legend(fontsize=8)
ax.set_title(f"Stage Weight Trajectories\n{LABEL}")
_save(fig, "06_stage_weight_trajectories.png")

# 07 Gamma trajectories
fig, ax = plt.subplots(figsize=(9, 5))
for col, ttl, colr in zip(["gamma1", "gamma2"], ["gamma1 (support attention)", "gamma2 (query attention)"], (C_MAIN, C_ALT)):
    ax.plot(range(0, MAX_EPOCHS + 1), [gamma_init] + [row[col] for row in r["epoch_log"]],
            marker="o", ms=3, label=ttl, color=colr)
ax.set_xlabel("Epoch"); ax.set_ylabel("Value"); ax.legend(fontsize=9)
ax.set_title(f"Self-Attention Gamma Trajectories\n{LABEL}")
_save(fig, "07_gamma_trajectories.png")

# 08 DW scalar parameters: fisher temperature and Fisher-vs-confidence mixing share (live vs EMA)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, ttl, init_v in zip(
        axes, ["dw_fisher_temp", "dw_mix_alpha"],
        ["DW fisher_temp", "DW Fisher share  sigmoid(mix_logit)"], [FISHER_TEMP, 0.5]):
    ax.plot(range(0, MAX_EPOCHS + 1), [snap[col] for snap in r["trajectory"]],
            marker="o", ms=3, color=C_MAIN, label="live model")
    ax.plot(range(0, MAX_EPOCHS + 1), [init_v] + [row[f"ema_{col}"] for row in r["epoch_log"]],
            ls="--", color=C_ALT, label="EMA model (evaluated)")
    ax.set_title(ttl); ax.set_xlabel("Epoch")
axes[0].legend(fontsize=9)
fig.suptitle(f"Dynamic-Weight Parameter Trajectories — {LABEL}")
_save(fig, "08_dw_parameter_trajectories.png")

# 09 Mean dynamic stage weights on the validation episodes, per epoch (EMA model)
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(5):
    m_ = np.array([row[f"dw_val_w{i+1}"]     for row in r["epoch_log"]])
    s_ = np.array([row[f"dw_val_w{i+1}_std"] for row in r["epoch_log"]])
    ax.plot(EPOCHS, m_, marker="o", ms=3, color=stage_colors[i], label=f"w{i+1} ({STAGE_NODES[i]})")
    ax.fill_between(EPOCHS, m_ - s_, m_ + s_, color=stage_colors[i], alpha=0.15)
ax.set_xlabel("Epoch"); ax.set_ylabel("Mean dynamic weight (±1 std over tasks)"); ax.legend(fontsize=8)
ax.set_title(f"Task-Adaptive Dynamic Stage Weights (validation episodes)\n{LABEL}")
_save(fig, "09_dw_dynamic_weights_per_epoch.png")

# 10 Test-time dynamic weights vs the global softplus prior
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(5); w = 0.38
prior = [r["params_best_checkpoint"][f"prior_w{i+1}"] for i in range(5)]
ax.bar(x - w/2, prior, w, color="#9ecae1", label="global prior softplus(w)")
ax.bar(x + w/2, r["dw_test_mean"], w, yerr=r["dw_test_std"], capsize=4, color=C_MAIN,
       label=f"dynamic weight (mean ± std, {N_TEST_TASKS} test tasks)")
ax.set_xticks(x); ax.set_xticklabels([f"w{i+1}\n{STAGE_NODES[i]}" for i in range(5)])
ax.set_ylabel("Stage weight"); ax.legend(fontsize=8)
ax.set_title(f"Test-Time Dynamic Weights\n{LABEL}"); ax.grid(axis="x", visible=False)
_save(fig, "10_dw_test_time_weights.png")

print(f"\nAll plots saved to {PLOT_DIR}")

  saved 01_test_accuracy.png
  saved 02_per_episode_test_accuracy.png
  saved 03_accuracy_vs_epoch.png
  saved 04_loss_vs_epoch.png
  saved 05_val_accuracy_vs_epoch.png
  saved 06_stage_weight_trajectories.png
  saved 07_gamma_trajectories.png
  saved 08_dw_parameter_trajectories.png
  saved 09_dw_dynamic_weights_per_epoch.png
  saved 10_dw_test_time_weights.png

All plots saved to /kaggle/working/results/plots


In [13]:
# ============================================================================
# Cell 13 — Experiment summary text + ZIP (includes the complete .pth model; no auto-download)
# ============================================================================
import zipfile, datetime

r   = RESULTS[KEY]
cfg = EXPERIMENT_CONFIGS[KEY]
_ep = np.array(r["per_episode_acc"])
_ci = 1.96 * _ep.std(ddof=1) / math.sqrt(len(_ep))

lines = []
lines.append("=" * 72)
lines.append("MSENet Final Model — Dynamic Weights + LayerNorm + EMA (Euclidean, no GeM)")
lines.append(f"Generated: {datetime.datetime.now().isoformat()}")
lines.append("=" * 72)
lines.append("")
lines.append("*" * 72)
lines.append(f"*  FINAL TEST ACCURACY ({TEST_WAY}-way {SHOT}-shot, {N_TEST_TASKS} episodes): {r['test_acc']:.2f}%  (95% CI ±{_ci:.2f})")
lines.append(f"*  Best validation accuracy: {r['best_val_acc']:.2f}% @ epoch {r['best_epoch']}")
lines.append("*" * 72)
lines.append("")
lines.append("Distance: Euclidean (fixed)   |   Pooling: global average pooling (no GeM)")
lines.append("")
lines.append("Hyperparameters:")
for k, v in [
    ("Epochs x Episodes",       f"{MAX_EPOCHS} x {N_TRAINING_EPISODES}"),
    ("Evaluation protocol",     f"{TEST_WAY}-way {SHOT}-shot"),
    ("Val / Test tasks",        f"{N_VALIDATION_TASKS} / {N_TEST_TASKS}"),
    ("backbone_lr",             backbone_lr),
    ("weights_lr",              weights_lr),
    ("dynamic_gate_lr",         dynamic_gate_lr),
    ("DW gate scale",           DYNAMIC_GATE_SCALE),
    ("DW fisher_temp init",     FISHER_TEMP),
    ("EMA beta",                EMA_BETA),
    ("STAGE_WEIGHTS init",      STAGE_WEIGHTS),
    ("gamma_init",              gamma_init),
    ("AMP",                     USE_AMP),
    ("Seeds (train/val/test)",  f"{TRAIN_EPISODE_SEED}/{VAL_EPISODE_SEED}/{TEST_EPISODE_SEED}"),
]:
    lines.append(f"  {k:30s} = {v}")
lines.append("")
lines.append(f"Configuration: [{KEY}] {cfg['description']}  layernorm={cfg['use_layernorm']}  "
             f"ema={cfg['use_ema']}  gem=False  gammalearnable={cfg['gammalearnable']}")
lines.append("")
lines.append(f"Training time: {r['train_time_total_s']/60:.1f} min   |   Inference: {r['infer_ms_per_episode']:.1f} ms/episode   |   "
             f"Params: {r['n_total']:,}")
lines.append(f"Test loss: {r['test_loss']:.4f}")
lines.append("")
lines.append("Dynamic weights on test episodes (mean over tasks):")
lines.append(f"  {[round(float(x), 4) for x in r['dw_test_mean']]}")
lines.append("")
lines.append(f"Saved model: model/{CKPT_PATH.name}")
lines.append("  (load with load_checkpoint_model(path) from Cell 7; contains weights + architecture config + results)")
lines.append("")
lines.append("Environment:")
for k, v in ENV_INFO.items():
    lines.append(f"  {k}: {v}")

(RESULT_DIR / "experiment_summary.txt").write_text("\n".join(lines))
print("Saved: experiment_summary.txt")
print()
for line in lines:
    print(line)

# Verify exactly one checkpoint (.pth) is present and non-trivial
pths = [p for p in RESULT_DIR.rglob("*") if p.suffix in (".pth", ".pt", ".ckpt", ".bin")]
assert len(pths) == 1 and pths[0] == CKPT_PATH, f"Expected exactly {CKPT_PATH}, found: {pths}"
assert CKPT_PATH.stat().st_size > 1e6, "checkpoint file suspiciously small"

# Create ZIP (results + plots + CSVs + the .pth model)
zip_path = WORK_DIR / "msenet_dw_ln_ema_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(RESULT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(RESULT_DIR))

print(f"\nFiles in ZIP ({zip_path.stat().st_size/1e6:.2f} MB):")
for p in sorted(RESULT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RESULT_DIR)}  ({p.stat().st_size/1024:.1f} KB)")
print(f"\nZIP saved at: {zip_path}")
print("Download it from Kaggle's Output / Files panel (no auto-download).")
print()
print("=" * 56)
print(f"FINAL TEST ACCURACY ({TEST_WAY}-way {SHOT}-shot): {r['test_acc']:.2f}%")
print("=" * 56)

Saved: experiment_summary.txt

MSENet Final Model — Dynamic Weights + LayerNorm + EMA (Euclidean, no GeM)
Generated: 2026-11-05T03:36:51.845403

************************************************************************
*  FINAL TEST ACCURACY (5-way 5-shot, 600 episodes): 85.33%  (95% CI ±0.49)
*  Best validation accuracy: 84.26% @ epoch 1
************************************************************************

Distance: Euclidean (fixed)   |   Pooling: global average pooling (no GeM)

Hyperparameters:
  Epochs x Episodes              = 10 x 360
  Evaluation protocol            = 5-way 5-shot
  Val / Test tasks               = 600 / 600
  backbone_lr                    = 0.0001
  weights_lr                     = 0.005
  dynamic_gate_lr                = 0.001
  DW gate scale                  = 0.5
  DW fisher_temp init            = 2.0
  EMA beta                       = 0.99
  STAGE_WEIGHTS init             = (1.0, 1.1, 1.2, 1.3, 1.4)
  gamma_init                     = 0.2
  AMP         